# Task 9 – Hybrid Recommendation Model

**Objective:** Combine content-based filtering (what a talk is *about*) and collaborative filtering (what *similar users* liked) to recommend TED Talks for a user.

> **Note:** The user interaction data is **simulated** (created in Task 7).

In [1]:
import pandas as pd                                         # tables
import numpy as np                                          # number helpers
from sklearn.feature_extraction.text import TfidfVectorizer # turns text into numbers
from sklearn.metrics.pairwise import cosine_similarity      # measures how alike two things are

In [2]:
# Step 1: load the Task 7 interaction data
interactions = pd.read_csv("../data/interaction/user_interactions.csv")
interactions["talk_id"] = pd.to_numeric(interactions["talk_id"])

# Step 2: check the TED dataset has the columns we need
content_path = "../data/processed/ted_talks_cleaned.csv"
header = pd.read_csv(content_path, nrows=0).columns.tolist()
needed_cols = ["talk_id", "title", "description", "topics"]
missing_cols = [c for c in needed_cols if c not in header]

print("Interaction data shape:", interactions.shape)
print("Columns in cleaned TED file:", header)
print("Missing needed columns:", missing_cols)

if missing_cols:
    print("\n>>> Some columns are missing. Send me this output before going on.")
else:
    talks = pd.read_csv(content_path, usecols=needed_cols)
    talks["talk_id"] = pd.to_numeric(talks["talk_id"])
    talks = talks.drop_duplicates(subset="talk_id").reset_index(drop=True)
    print("\nTalks loaded:", talks.shape)
    display(talks.head(3))

Interaction data shape: (181, 7)
Columns in cleaned TED file: ['talk_id', 'title', 'speaker_1', 'all_speakers', 'occupations', 'about_speakers', 'views', 'recorded_date', 'published_date', 'event', 'native_lang', 'available_lang', 'comments', 'duration', 'topics', 'related_talks', 'url', 'description', 'transcript', 'title_clean', 'topics_clean', 'description_clean', 'combined_text']
Missing needed columns: []

Talks loaded: (4005, 4)


,talk_id,title,topics,description
0,1,Averting the climate crisis,"['alternative energy', 'cars', 'climate change...",With the same humor and humanity he exuded in ...
1,92,The best stats you've ever seen,"['Africa', 'Asia', 'Google', 'demo', 'economic...",You've never seen data presented like this. Wi...
2,7,Simplicity sells,"['computers', 'entertainment', 'interface desi...",New York Times columnist David Pogue takes aim...


## Part A: Content-Based Model (rebuilt from Task 6)
Using **Experiment 3 (Title + Description + Tags)**, my final choice in Task 6. TF-IDF is built **once**, not on every call. A user's taste profile = the talks they liked. A talk's content score = its average similarity to those liked talks.

In [3]:
# empty text becomes "" so joining never breaks
for col in ["title", "description", "topics"]:
    talks[col] = talks[col].fillna("")

# Experiment 3 from Task 6: title + description + tags
talks["hybrid_text"] = talks["title"] + " " + talks["description"] + " " + talks["topics"]

# turn text into numbers (done once)
tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(talks["hybrid_text"])

# quick lookup: talk_id -> row number in the TF-IDF matrix
talk_index = pd.Series(talks.index, index=talks["talk_id"])
title_lookup = talks.set_index("talk_id")["title"]

print("TF-IDF matrix shape (talks x words):", tfidf_matrix.shape)

TF-IDF matrix shape (talks x words): (4005, 21509)


In [4]:
def get_seen_talks(user_id):
    # talks the user already rated or watched (never recommend these again)
    rows = interactions[interactions["user_id"] == user_id]
    seen = rows[(rows["watched"] == 1) | (rows["rating"].notna())]
    return set(seen["talk_id"])

def get_liked_talks(user_id):
    # talks the user clearly enjoyed: rating 4 or 5, or a "like" reaction
    rows = interactions[interactions["user_id"] == user_id]
    liked = rows[(rows["rating"] >= 4) | (rows["reaction"] == "like")]
    return list(liked["talk_id"].unique())

def get_content_scores(user_id):
    # score EVERY unseen talk by how much it reads like the talks this user liked
    liked = get_liked_talks(user_id)
    liked = [t for t in liked if t in talk_index.index]    # keep only talks that exist in the TED file
    if len(liked) == 0:
        return pd.Series(dtype=float)                      # no liked talks = no profile

    liked_rows = talk_index[liked].values                  # matrix rows of the liked talks
    similarity = cosine_similarity(tfidf_matrix[liked_rows], tfidf_matrix)  # liked talks vs all talks
    average_similarity = similarity.mean(axis=0)           # average across liked talks

    scores = pd.Series(average_similarity, index=talks["talk_id"])
    seen = [t for t in get_seen_talks(user_id) if t in scores.index]
    return scores.drop(labels=seen).sort_values(ascending=False)

def recommend_content_for_user(user_id, top_n=5):
    if user_id not in interactions["user_id"].values:
        print(f"User ID '{user_id}' not found.")
        return pd.DataFrame(columns=["talk_id", "title", "content_score"])
    scores = get_content_scores(user_id).head(top_n)
    result = scores.reset_index()
    result.columns = ["talk_id", "content_score"]
    result["title"] = result["talk_id"].map(title_lookup)
    result["content_score"] = result["content_score"].round(3)
    return result[["talk_id", "title", "content_score"]]

## Part B: Collaborative Filtering Model (from Task 8)
Same idea as Task 8: find users with similar ratings and take a similarity-weighted average of their ratings. This version scores every unseen talk so the hybrid can use all of them.

In [5]:
# user-item matrix: rows = users, columns = talks, values = ratings
user_item_matrix = interactions.pivot_table(index="user_id", columns="talk_id", values="rating", aggfunc="mean")

# user similarity matrix (missing ratings filled with 0, as in Task 8)
user_similarity_df = pd.DataFrame(cosine_similarity(user_item_matrix.fillna(0)),
                                  index=user_item_matrix.index,
                                  columns=user_item_matrix.index)

def get_similar_users(user_id, n=5):
    scores = user_similarity_df[user_id].drop(user_id)     # drop the user themselves
    return scores.sort_values(ascending=False).head(n)

def get_cf_scores(user_id, n_neighbors=10):
    # predicted rating (1 to 5) for every talk the user has not seen
    if user_id not in user_item_matrix.index:
        return pd.Series(dtype=float)
    similar_users = get_similar_users(user_id, n_neighbors)
    similar_users = similar_users[similar_users > 0]
    if similar_users.empty:
        return pd.Series(dtype=float)

    seen = get_seen_talks(user_id)
    candidates = [t for t in user_item_matrix.columns if t not in seen]
    neighbor_ratings = user_item_matrix.loc[similar_users.index, candidates]

    weighted_sum = neighbor_ratings.fillna(0).mul(similar_users, axis=0).sum()
    similarity_sum = neighbor_ratings.notna().mul(similar_users, axis=0).sum()
    predicted = (weighted_sum / similarity_sum).dropna()
    return predicted.sort_values(ascending=False)

def recommend_cf_for_user(user_id, top_n=5):
    if user_id not in user_item_matrix.index:
        print(f"User ID '{user_id}' not found.")
        return pd.DataFrame(columns=["talk_id", "title", "predicted_rating"])
    scores = get_cf_scores(user_id).head(top_n)
    result = scores.reset_index()
    result.columns = ["talk_id", "predicted_rating"]
    result["title"] = result["talk_id"].map(title_lookup)
    result["predicted_rating"] = result["predicted_rating"].round(2)
    return result[["talk_id", "title", "predicted_rating"]]

print("User-item matrix:", user_item_matrix.shape, "| Similarity matrix:", user_similarity_df.shape)

User-item matrix: (24, 24) | Similarity matrix: (24, 24)


## Part C: Hybrid Model
**Steps:** (1) take the top 20 talks from each model, (2) remove duplicates so each talk appears once, (3) put both scores on a 0 to 1 scale, (4) combine them:

**hybrid score = cf_weight × collaborative score + (1 − cf_weight) × content score**

- Collaborative score: predicted rating ÷ 5.
- Content score: divided by the highest content score, so the best content match = 1.
- A talk the collaborative model has never seen (it only knows 24 talks) gets a collaborative score of 0, but can still win on content.

In [6]:
def recommend_hybrid(user_id, top_n=5, cf_weight=0.5, pool_size=20, show_info=False):
    empty_table = pd.DataFrame(columns=["talk_id", "title", "cf_score", "content_score", "hybrid_score", "found_by"])

    # 0. checks
    if user_id not in interactions["user_id"].values:
        print(f"User ID '{user_id}' not found. Please enter a valid user ID.")
        return empty_table
    if not 0 <= cf_weight <= 1:
        print("cf_weight must be between 0 and 1.")
        return empty_table

    # 1. get scores from both models
    cf_scores = get_cf_scores(user_id)
    content_scores = get_content_scores(user_id)
    if cf_scores.empty and content_scores.empty:
        print(f"No recommendations possible for user {user_id} (no history). This is the cold-start problem.")
        return empty_table

    cf_top = cf_scores.head(pool_size)
    content_top = content_scores.head(pool_size)

    # 2. join both candidate lists and remove duplicates
    all_candidates = pd.concat([pd.Series(cf_top.index), pd.Series(content_top.index)])
    unique_candidates = all_candidates.drop_duplicates().tolist()
    if show_info:
        print(f"Candidates before removing duplicates: {len(all_candidates)} | after: {len(unique_candidates)}")

    # 3. look up both scores for every candidate (0 if a model has no score for it)
    cf_raw = pd.Series([cf_scores.get(t, 0.0) for t in unique_candidates], index=unique_candidates)
    content_raw = pd.Series([content_scores.get(t, 0.0) for t in unique_candidates], index=unique_candidates)

    # 4. put both on a 0 to 1 scale
    cf_scaled = cf_raw / 5.0
    best_content = content_scores.max() if len(content_scores) > 0 else 1.0
    content_scaled = content_raw / best_content if best_content > 0 else content_raw

    # 5. weighted combination
    hybrid_score = cf_weight * cf_scaled + (1 - cf_weight) * content_scaled

    # 6. note which model(s) suggested each talk
    def found_by(talk_id):
        in_cf = talk_id in cf_top.index
        in_content = talk_id in content_top.index
        if in_cf and in_content:
            return "Both"
        if in_cf:
            return "Collaborative only"
        return "Content only"

    result = pd.DataFrame({
        "talk_id": unique_candidates,
        "cf_score": cf_scaled.values.round(3),
        "content_score": content_scaled.values.round(3),
        "hybrid_score": hybrid_score.values.round(3),
        "found_by": [found_by(t) for t in unique_candidates],
    })
    result["title"] = result["talk_id"].map(title_lookup)

    # 7. rank highest first and keep the top N
    result = result.sort_values("hybrid_score", ascending=False).head(top_n).reset_index(drop=True)
    return result[["talk_id", "title", "cf_score", "content_score", "hybrid_score", "found_by"]]

## Part D: Comparing the Three Models and Testing

In [7]:
def compare_models(user_id, top_n=5):
    print("=" * 70)
    print(f"USER {user_id}  |  talks liked: {len(get_liked_talks(user_id))}  |  talks seen: {len(get_seen_talks(user_id))}")
    print("=" * 70)

    cf_result = recommend_cf_for_user(user_id, top_n)
    content_result = recommend_content_for_user(user_id, top_n)
    hybrid_result = recommend_hybrid(user_id, top_n, show_info=True)

    print("\n1) Collaborative filtering only:")
    display(cf_result)
    print("2) Content-based only:")
    display(content_result)
    print("3) HYBRID (50% collaborative + 50% content):")
    display(hybrid_result)

    cf_ids = set(cf_result["talk_id"])
    content_ids = set(content_result["talk_id"])
    hybrid_ids = set(hybrid_result["talk_id"])
    print(f"Hybrid talks that also came from collaborative top {top_n}: {len(hybrid_ids & cf_ids)}")
    print(f"Hybrid talks that also came from content top {top_n}: {len(hybrid_ids & content_ids)}")

compare_models(1)

USER 1  |  talks liked: 5  |  talks seen: 8
Candidates before removing duplicates: 35 | after: 34

1) Collaborative filtering only:


,talk_id,title,predicted_rating
0,2476,Your kids might live on Mars. Here's how they'...,5.00
1,685,The thrilling potential of SixthSense technology,4.06
2,66,Do schools kill creativity?,4.00
3,1042,The power of vulnerability,3.55
4,2452,This country isn't just carbon neutral -- it's...,3.45


2) Content-based only:


,talk_id,title,content_score
0,14615,SpaceX's plan to fly you across the globe in 3...,0.082
1,1695,"The mind behind Tesla, SpaceX, SolarCity ...",0.081
2,2476,Your kids might live on Mars. Here's how they'...,0.070
3,833,Glowing life in an underwater world,0.062
4,10807,A funny look at the unintended consequences of...,0.062


3) HYBRID (50% collaborative + 50% content):


,talk_id,title,cf_score,content_score,hybrid_score,found_by
0,2476,Your kids might live on Mars. Here's how they'...,1.000,0.854,0.927,Both
1,685,The thrilling potential of SixthSense technology,0.811,0.240,0.526,Collaborative only
2,14615,SpaceX's plan to fly you across the globe in 3...,0.000,1.000,0.500,Content only
3,1695,"The mind behind Tesla, SpaceX, SolarCity ...",0.000,0.998,0.499,Content only
4,2452,This country isn't just carbon neutral -- it's...,0.689,0.288,0.489,Collaborative only


Hybrid talks that also came from collaborative top 5: 3
Hybrid talks that also came from content top 5: 3


In [8]:
compare_models(8)

USER 8  |  talks liked: 4  |  talks seen: 10
Candidates before removing duplicates: 32 | after: 32

1) Collaborative filtering only:


,talk_id,title,predicted_rating
0,2774,The future we're building -- and boring,4.70
1,1386,Questions no one knows the answers to,4.52
2,206,Underwater astonishments,4.31
3,2361,How to stay calm when you know you'll be stressed,3.00
4,2217,The price of shame,2.73


2) Content-based only:


,talk_id,title,content_score
0,10190,How teachers can help kids find their politica...,0.119
1,2615,Help for kids the education system ignores,0.113
2,865,Bring on the learning revolution!,0.112
3,2582,How to raise successful kids -- without over-p...,0.101
4,949,The child-driven education,0.098


3) HYBRID (50% collaborative + 50% content):


,talk_id,title,cf_score,content_score,hybrid_score,found_by
0,1386,Questions no one knows the answers to,0.904,0.162,0.533,Collaborative only
1,2774,The future we're building -- and boring,0.941,0.065,0.503,Collaborative only
2,10190,How teachers can help kids find their politica...,0.000,1.000,0.500,Content only
3,2615,Help for kids the education system ignores,0.000,0.946,0.473,Content only
4,865,Bring on the learning revolution!,0.000,0.940,0.470,Content only


Hybrid talks that also came from collaborative top 5: 2
Hybrid talks that also came from content top 5: 3


In [9]:
test_users = [1, 3, 8]
hybrid_results = {}

for u in test_users:
    print(f"\n=== Hybrid recommendations for user {u} ===")
    hybrid_results[u] = recommend_hybrid(u, top_n=5)
    display(hybrid_results[u])

print("\n----- AUTOMATIC CHECKS -----")
for u, rec in hybrid_results.items():
    seen = get_seen_talks(u)
    print(f"User {u}: got {len(rec)} recs"
          f" | no seen talks: {len(set(rec['talk_id']) & seen) == 0}"
          f" | no duplicates: {rec['talk_id'].is_unique}"
          f" | sorted high to low: {rec['hybrid_score'].is_monotonic_decreasing}")

different = len({tuple(r["talk_id"]) for r in hybrid_results.values()}) > 1
print("Different users got different recommendations:", different)

print("\n--- Invalid user test ---")
_ = recommend_hybrid("NOT_A_USER")
_ = recommend_hybrid(99999)


=== Hybrid recommendations for user 1 ===


,talk_id,title,cf_score,content_score,hybrid_score,found_by
0,2476,Your kids might live on Mars. Here's how they'...,1.000,0.854,0.927,Both
1,685,The thrilling potential of SixthSense technology,0.811,0.240,0.526,Collaborative only
2,14615,SpaceX's plan to fly you across the globe in 3...,0.000,1.000,0.500,Content only
3,1695,"The mind behind Tesla, SpaceX, SolarCity ...",0.000,0.998,0.499,Content only
4,2452,This country isn't just carbon neutral -- it's...,0.689,0.288,0.489,Collaborative only



=== Hybrid recommendations for user 3 ===


,talk_id,title,cf_score,content_score,hybrid_score,found_by
0,1386,Questions no one knows the answers to,0.800,0.316,0.558,Collaborative only
1,1042,The power of vulnerability,0.703,0.312,0.508,Collaborative only
2,1695,"The mind behind Tesla, SpaceX, SolarCity ...",0.000,1.000,0.500,Content only
3,14615,SpaceX's plan to fly you across the globe in 3...,0.000,0.985,0.492,Content only
4,66,Do schools kill creativity?,0.836,0.111,0.473,Collaborative only



=== Hybrid recommendations for user 8 ===


,talk_id,title,cf_score,content_score,hybrid_score,found_by
0,1386,Questions no one knows the answers to,0.904,0.162,0.533,Collaborative only
1,2774,The future we're building -- and boring,0.941,0.065,0.503,Collaborative only
2,10190,How teachers can help kids find their politica...,0.000,1.000,0.500,Content only
3,2615,Help for kids the education system ignores,0.000,0.946,0.473,Content only
4,865,Bring on the learning revolution!,0.000,0.940,0.470,Content only



----- AUTOMATIC CHECKS -----
User 1: got 5 recs | no seen talks: True | no duplicates: True | sorted high to low: True
User 3: got 5 recs | no seen talks: True | no duplicates: True | sorted high to low: True
User 8: got 5 recs | no seen talks: True | no duplicates: True | sorted high to low: True
Different users got different recommendations: True

--- Invalid user test ---
User ID 'NOT_A_USER' not found. Please enter a valid user ID.
User ID '99999' not found. Please enter a valid user ID.


In [10]:
# same user, three different weights: 0 = content only, 0.5 = equal mix, 1 = collaborative only
for w in [0.0, 0.5, 1.0]:
    print(f"\n--- User 1 | cf_weight = {w} ---")
    display(recommend_hybrid(1, top_n=5, cf_weight=w)[["talk_id", "title", "hybrid_score", "found_by"]])


--- User 1 | cf_weight = 0.0 ---


,talk_id,title,hybrid_score,found_by
0,14615,SpaceX's plan to fly you across the globe in 3...,1.000,Content only
1,1695,"The mind behind Tesla, SpaceX, SolarCity ...",0.998,Content only
2,2476,Your kids might live on Mars. Here's how they'...,0.854,Both
3,833,Glowing life in an underwater world,0.762,Content only
4,10807,A funny look at the unintended consequences of...,0.755,Content only



--- User 1 | cf_weight = 0.5 ---


,talk_id,title,hybrid_score,found_by
0,2476,Your kids might live on Mars. Here's how they'...,0.927,Both
1,685,The thrilling potential of SixthSense technology,0.526,Collaborative only
2,14615,SpaceX's plan to fly you across the globe in 3...,0.500,Content only
3,1695,"The mind behind Tesla, SpaceX, SolarCity ...",0.499,Content only
4,2452,This country isn't just carbon neutral -- it's...,0.489,Collaborative only



--- User 1 | cf_weight = 1.0 ---


,talk_id,title,hybrid_score,found_by
0,2476,Your kids might live on Mars. Here's how they'...,1.000,Both
1,685,The thrilling potential of SixthSense technology,0.811,Collaborative only
2,66,Do schools kill creativity?,0.800,Collaborative only
3,1042,The power of vulnerability,0.711,Collaborative only
4,2452,This country isn't just carbon neutral -- it's...,0.689,Collaborative only


## Observations

**1. Comparison for User 1 (liked 5 talks, seen 8)**
- Collaborative filtering recommended Mars, SixthSense, "Do schools kill creativity?", "The power of vulnerability" and the carbon neutral country talk. Its top predicted rating was 5.00.
- Content-based recommended mostly SpaceX, Tesla and underwater-themed talks, with content scores only between 0.062 and 0.082. This is because they read like the talks the user liked.
- Only one talk, "Your kids might live on Mars" (talk 2476), was in both top 5 lists. The hybrid ranked it first with a score of 0.927 (cf 1.000, content 0.854), and it was marked "Both".
- The hybrid top 5 had 3 talks from the collaborative top 5 and 3 from the content top 5. Talk 2476 counts in both numbers, which is why they add up to 6.
- Before duplicate removal there were 35 candidates and after it there were 34. So the duplicate removal worked and removed 1 repeated talk.

**2. Comparison for User 8 (liked 4 talks, seen 10)**
- Collaborative filtering recommended "The future we're building -- and boring", "Questions no one knows the answers to", "Underwater astonishments", and others.
- Content-based recommended only education and kids talks (talk 10190, 2615, 865, 2582, 949).
- This time the two models did not agree on anything. There were 32 candidates before and after removing duplicates, so no talk was repeated.
- The hybrid mixed both sides: 2 collaborative talks (1386 and 2774) and 3 content talks (10190, 2615, 865).

**3. Testing different users**
- Users 1, 3 and 8 all got 5 recommendations, with no already-seen talks, no duplicates, and scores sorted from high to low.
- Different users got different recommendation lists (checked automatically: True).
- Users 3 and 8 both got "Questions no one knows the answers to" first, which means their tastes are similar in the collaborative model.
- Users 1 and 3 both got the SpaceX talks (14615 and 1695) from the content side.
- Invalid users ("NOT_A_USER" and 99999) gave a clear "not found" message and did not crash.

**4. What the weight changed (User 1)**
- cf_weight = 0.0 gave the same 5 talks as content-based only (top: SpaceX talk 14615 with score 1.000).
- cf_weight = 1.0 gave the same 5 talks as collaborative only (top: Mars talk 2476 with score 1.000).
- cf_weight = 0.5 mixed both and had 2 collaborative-only, 2 content-only and 1 "Both" talk.
- Talk 2476 (Mars) was in the top 5 at every weight, so it is the strongest recommendation for this user.

**5. Main findings**
- The hybrid gives a more balanced list. It keeps the best talks from each model instead of trusting only one.
- A talk found by both models gets a much higher hybrid score (0.927) than any talk found by only one model (around 0.5).
- Content-only talks always have a collaborative score of 0, because the collaborative model only knows 24 talks. So they top out at 0.5 with a 50/50 weight, and ranks 2 to 5 end up very close together (for example 0.526, 0.500, 0.499, 0.489 for user 1).
- Changing the weight changes the list a lot, so the weight is an important setting.

## Questions and Answers

**1. What is a Hybrid Recommendation Model?**
A model that mixes two or more recommendation methods. Mine mixes content-based filtering (what a talk is about) with collaborative filtering (what similar users liked).

**2. Why combine them?**
Each one covers the other's weakness. Content-based can recommend talks nobody has rated yet, and collaborative can find surprising talks that don't read alike but that similar people enjoyed.

**3. How are the scores combined?**
Both scores are put on a 0 to 1 scale, then: hybrid = cf_weight × collaborative + (1 − cf_weight) × content. I used 0.5 and 0.5.

**4. Why remove duplicates?**
The same talk can appear in both lists. Without removing it, it would show twice in the final list.

**5. What does changing the weight do?**
A higher cf_weight follows similar users more. A lower one follows talk content more. At 0 it is content only, and at 1 it is collaborative only.

**6. What are the limitations of my hybrid?**
- The collaborative part only knows 24 talks, so most of the 4005 talks get a collaborative score of 0.
- The data is small and simulated.
- A brand-new user with no history gets nothing (cold start).
- The 50/50 weight was chosen by me, not learned from data.